# Flat $\Lambda$CDM with Particle Swarm Optimization

This notebook fits a spatially flat $\Lambda$CDM model to 31 cosmic-chronometer $H(z)$ measurements using a particle swarm optimizer implemented in this repository. The optimization minimizes

$$
\chi^2(H_0,\Omega_m)=\sum_i\left[\frac{H_{\rm model}(z_i)-H_i}{\sigma_{H,i}}\right]^2,
$$

with

$$
H(z)=H_0\sqrt{\Omega_m(1+z)^3+1-\Omega_m}.
$$

The full run uses 100 particles, 300 iterations, inertia $0.72$, cognitive/social coefficients $1.49$, and random seed 42.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from pso_cosmology import (
    load_cosmic_chronometers,
    h_flat_lcdm,
    h_nonflat_lcdm,
    h_flat_xcdm,
    h_nonflat_xcdm,
    run_pso,
    chi_square,
    model_statistics,
)

DATA_FILE = ROOT / "data" / "cosmic_chronometers_31.txt"
OUTPUT_DIR = Path(os.environ.get("PSO_OUTPUT_DIR", ROOT / "results" / "figures"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QUICK = os.environ.get("PSO_QUICK", "0") == "1"

z, H_obs, sigma_H = load_cosmic_chronometers(DATA_FILE)
print(f"Loaded {len(z)} cosmic-chronometer measurements spanning z={z.min():.3f}–{z.max():.3f}.")
print("Quick mode:", QUICK)


In [ ]:
def objective(theta):
    H0, omega_m = theta
    if not (50.0 <= H0 <= 100.0 and 0.1 <= omega_m <= 0.9):
        return np.inf
    return chi_square(h_flat_lcdm(z, H0, omega_m), H_obs, sigma_H)

settings = dict(
    n_particles=35 if QUICK else 100,
    n_iter=80 if QUICK else 300,
    inertia=0.72,
    cognitive=1.49,
    social=1.49,
    seed=42,
)
result = run_pso(objective, [(50.0, 100.0), (0.1, 0.9)], **settings)
H0_best, omega_m_best = result.best_position
stats = model_statistics(result.best_score, len(z), 2)

print(f"H0           = {H0_best:.6f} km s^-1 Mpc^-1")
print(f"Omega_m      = {omega_m_best:.6f}")
print(f"Omega_Lambda = {1.0-omega_m_best:.6f}")
print(f"chi2_min     = {stats.chi2:.6f}")
print(f"reduced chi2 = {stats.reduced_chi2:.6f}")

In [ ]:
z_grid = np.linspace(0.0, 2.05, 500)
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.errorbar(z, H_obs, yerr=sigma_H, fmt="o", capsize=3, label="Cosmic chronometers")
ax.plot(z_grid, h_flat_lcdm(z_grid, H0_best, omega_m_best), lw=2.5,
        label=rf"PSO best fit: $H_0={H0_best:.2f}$, $\Omega_m={omega_m_best:.3f}$")
ax.set(xlabel="Redshift z", ylabel=r"$H(z)$ [km s$^{-1}$ Mpc$^{-1}$]",
       title=r"Flat $\Lambda$CDM fit to cosmic-chronometer data")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "flat_lcdm_best_fit.png", dpi=220)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(result.best_score_history, lw=2.3)
ax.set(xlabel="PSO iteration", ylabel=r"Best $\chi^2$ found so far",
       title="Particle-swarm convergence")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "flat_lcdm_convergence.png", dpi=220)
plt.show()

In [ ]:
initial = result.initial_positions
final = result.final_positions
history = result.positions_history

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(initial[:,0], initial[:,1], s=25, alpha=0.30, label="Initial swarm")
ax.scatter(final[:,0], final[:,1], s=48, alpha=0.85, label="Final swarm")
step = max(1, len(history)//25)
for p in np.linspace(0, history.shape[1]-1, min(24, history.shape[1]), dtype=int):
    ax.plot(history[::step,p,0], history[::step,p,1], lw=0.8, alpha=0.35)
ax.scatter([H0_best],[omega_m_best], marker="x", s=100, linewidths=2.5, label="Global best")
ax.set(xlabel=r"$H_0$ [km s$^{-1}$ Mpc$^{-1}$]", ylabel=r"$\Omega_m$",
       title=r"PSO trajectories in $(H_0,\Omega_m)$")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "flat_lcdm_swarm_trajectories.png", dpi=220)
plt.show()

In [ ]:
H0_grid = np.linspace(50.0, 100.0, 180 if not QUICK else 80)
om_grid = np.linspace(0.1, 0.9, 180 if not QUICK else 80)
HH, OO = np.meshgrid(H0_grid, om_grid)
model = HH[...,None] * np.sqrt(OO[...,None]*(1.0+z)**3 + (1.0-OO[...,None]))
chi_map = np.sum(((model-H_obs)/sigma_H)**2, axis=-1)

fig, ax = plt.subplots(figsize=(8,5.5))
levels = np.linspace(np.nanmin(chi_map), min(np.nanpercentile(chi_map,75), 120), 30)
cont = ax.contourf(HH, OO, chi_map, levels=levels)
ax.scatter(result.initial_positions[:,0], result.initial_positions[:,1], s=12, alpha=0.30, label="Initial swarm")
ax.scatter(result.final_positions[:,0], result.final_positions[:,1], s=24, alpha=0.65, label="Final swarm")
ax.scatter([H0_best],[omega_m_best], marker="x", s=100, linewidths=2.5, label="Global best")
ax.set(xlabel=r"$H_0$ [km s$^{-1}$ Mpc$^{-1}$]", ylabel=r"$\Omega_m$",
       title=r"$\chi^2$ landscape and PSO swarm")
ax.legend()
fig.colorbar(cont, ax=ax, label=r"$\chi^2$")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "flat_lcdm_chi2_landscape.png", dpi=220)
plt.show()